In [16]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
!mkdir AISHELL3-audio

In [9]:
with open('train/label_train-set.txt') as fopen:
    d = fopen.read().split('\n')
d = [d_.split('|') for d_ in d if '|' in d_]
len(d)

63262

In [34]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        try:
            speaker = row[0][:7]
            f = f'train/wav/{speaker}/{row[0]}.wav'
            t = row[2]
            if len(t) < 2:
                continue

            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue

            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('AISHELL3-audio', audio_filename)
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"AISHELL3_train_{speaker}"
            })
        except Exception as e:
            print(e)
            pass
    return data

In [35]:
data = loop((d[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 22.65it/s]


In [37]:
data = multiprocessing(d, loop, cores = 20)

100%|██████████| 3163/3163 [03:30<00:00, 15.01it/s]


In [39]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'AISHELL3-audio/train_wav_SSB0043_SSB00430444.mp3',
 'text': '持起%红缨枪%追赶%对方%半公里$',
 'speaker': 'AISHELL3_train_SSB0043'}

In [40]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'AISHELL3')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 51.56ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 2.52MB / 2.52MB, 12.6MB/s  
Processing Files (1 / 1): 100%|██████████| 2.52MB / 2.52MB, 6.28MB/s  
New Data Upload: 100%|██████████| 2.52MB / 2.52MB, 6.28MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.26 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/fc1b993172ec910bb602665a049e2fef35292958', commit_message='Upload dataset', commit_description='', oid='fc1b993172ec910bb602665a049e2fef35292958', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [41]:
audio_files = [d['audio_filename'] for d in data]

with open('AISHELL3-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [44]:
folders = glob('AISHELL3-audio*')
folders = [f for f in folders if '.' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

AISHELL3-audio_neucodec
AISHELL3-audio


In [45]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('AISHELL3-audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   2%|▏         | 34.1MB / 1.98GB,   ???B/s  
Processing Files (0 / 1):   9%|▉         |  180MB / 1.98GB,  728MB/s  
Processing Files (0 / 1):  15%|█▌        |  301MB / 1.98GB,  668MB/s  
Processing Files (0 / 1):  22%|██▏       |  432MB / 1.98GB,  665MB/s  
Processing Files (0 / 1):  28%|██▊       |  550MB / 1.98GB,  645MB/s  
Processing Files (0 / 1):  35%|███▍      |  691MB / 1.98GB,  657MB/s  
Processing Files (0 / 1):  37%|███▋      |  738MB / 1.98GB,  587MB/s  
Processing Files (0 / 1):  44%|████▍     |  877MB / 1.98GB,  602MB/s  
Processing Files (0 / 1):  50%|█████     |  994MB / 1.98GB,  599MB/s  
Processing Files (0 / 1):  57%|█████▋    | 1.12GB / 1.98GB,  606MB/s  
Processing Files (0 / 1):  64%|██████▎   | 1.26GB / 1.98GB,  613MB/s  
Processing Files (0 / 1):  69%|██████▊   | 1.35GB / 1.98GB,  600MB/s  
Processing Files (0 / 1):  75%|███████▍  | 1.47GB / 1.98GB,  599MB/s  
Processing